# KV-Cache Optimization: Distributed AI Inference Strategy

The KV (Key-Value) cache is central to LLM inference performance. Every token
generated requires reading the full KV-cache from GPU memory, and the cache
grows with context length and concurrent requests. Managing this cache well
is the single biggest lever for improving throughput, latency, and cost.

This notebook covers every KV-cache optimization available in llm-d:

1. **Baseline** - no cache optimization (for comparison)
2. **Prefix-cache-aware routing** - route requests to replicas with warm caches
3. **Precise prefix cache indexer** - fine-grained cache tracking at the block level
4. **CPU offloading tier** - spill cold KV-cache entries to CPU memory
5. **Disk offloading tier** - further spill to NVMe for massive capacity

We measure each optimization's impact and compare them all at the end.

## The KV-Cache Lifecycle in LLM Inference

Understanding the KV-cache lifecycle is essential before optimizing it.

### What Is the KV-Cache?

During the prefill phase, the model computes **Key** and **Value** tensors
for every attention layer and every input token. These tensors are stored in
GPU memory as the KV-cache. During the decode phase, each new token attends
to all previous tokens via the KV-cache.

### Cache Size Per Request

For a typical model like Llama-3.1-70B with 80 layers and 8192 context length:

```
KV-cache per token = 2 (K+V) x 80 layers x 8192 head_dim x 2 bytes (FP16)
                   = ~2.6 MB per token

Full context (8K tokens) = ~20 GB per request
```

With 80 GB of GPU memory, you can serve only 3-4 concurrent 8K-context
requests on a single GPU. The KV-cache is usually the binding constraint,
not compute.

### The Lifecycle

```
1. Request arrives
   --> Allocate KV-cache blocks on GPU

2. Prefill phase
   --> Fill KV-cache with all input token K/V tensors

3. Decode phase (token by token)
   --> Read entire KV-cache each step
   --> Append one new K/V entry per step

4. Request completes
   --> Free KV-cache blocks (or keep for prefix caching)
```

Without optimization, step 4 frees the cache immediately. But if another
request arrives with the same prompt prefix, we would need to recompute the
entire prefill from scratch. This is where prefix caching helps.

In [ ]:
# Setup: deploy llm-d with a baseline configuration (no cache optimization)
import os
import time
import json
import subprocess

os.environ["NAMESPACE"] = "llm-d-kvcache"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"

# Clone repo if needed
!git clone https://github.com/llm-d/llm-d-production-stack.git 2>/dev/null || true

# Create namespace
!kubectl create namespace $NAMESPACE --dry-run=client -o yaml | kubectl apply -f -

print(f"Namespace: {os.environ['NAMESPACE']}")
print(f"Model: {os.environ['MODEL_NAME']}")
print("\nWe will deploy 4 replicas to observe routing differences.")

## Configuration 1: Baseline (No Cache Optimization)

First, we deploy with random routing and no prefix caching. This is the
worst-case scenario and serves as our performance baseline.

In [ ]:
# Deploy baseline: random routing, no prefix caching
baseline_epp_config = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-config
  namespace: llm-d-kvcache
data:
  config.yaml: |
    scoring:
      - name: random-scorer
        weight: 1.0
    prefixCache:
      enabled: false
"""

with open("/tmp/baseline-epp-config.yaml", "w") as f:
    f.write(baseline_epp_config)

!kubectl apply -f /tmp/baseline-epp-config.yaml

# Deploy model server with prefix caching disabled
!kubectl apply -k llm-d-production-stack/guides/optimized-baseline/model-server/ -n $NAMESPACE
!kubectl apply -k llm-d-production-stack/guides/optimized-baseline/router/ -n $NAMESPACE

# Scale to 4 replicas
!kubectl scale deployment/vllm --replicas=4 -n $NAMESPACE

print("Waiting for all pods...")
!kubectl wait --for=condition=ready pod -l app=vllm -n $NAMESPACE --timeout=600s
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s

print("\nBaseline deployment ready (random routing, no prefix cache).")

In [ ]:
# Measure baseline performance
# We send 20 requests with the same system prompt to simulate a chatbot workload.
# With random routing, most requests will NOT get a prefix cache hit.

import statistics

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d-kvcache "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

SYSTEM_PROMPT = (
    "You are a helpful AI assistant specializing in cloud-native infrastructure. "
    "You provide clear, concise answers about Kubernetes, containers, and "
    "distributed systems. Always include practical examples when possible."
)

questions = [
    "What is a pod?",
    "Explain deployments.",
    "What is a service mesh?",
    "How does HPA work?",
    "What is a DaemonSet?",
    "Explain init containers.",
    "What is a StatefulSet?",
    "How do ConfigMaps work?",
    "What is a PersistentVolume?",
    "Explain network policies.",
    "What is an Ingress?",
    "How does RBAC work?",
    "What is a CRD?",
    "Explain operators.",
    "What is a sidecar?",
    "How does DNS work in K8s?",
    "What is a namespace?",
    "Explain resource quotas.",
    "What is a Job?",
    "How does rolling update work?",
]

def run_benchmark(label, gateway_ip, questions, system_prompt):
    """Send requests and measure TTFT for each."""
    ttft_times = []
    for i, q in enumerate(questions):
        payload = json.dumps({
            "model": "Qwen/Qwen3-32B",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": q},
            ],
            "max_tokens": 50,
            "stream": True,
        })
        start = time.time()
        proc = subprocess.Popen(
            ["curl", "-s", "-N", f"http://{gateway_ip}:8080/v1/chat/completions",
             "-H", "Content-Type: application/json", "-d", payload],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        for line in proc.stdout:
            if b"data:" in line and b"[DONE]" not in line:
                ttft = time.time() - start
                ttft_times.append(ttft)
                break
        proc.wait()

    if ttft_times:
        p50 = statistics.median(ttft_times)
        p99 = sorted(ttft_times)[int(len(ttft_times) * 0.99)]
        print(f"  {label}:")
        print(f"    Requests:  {len(ttft_times)}")
        print(f"    TTFT p50:  {p50:.3f}s")
        print(f"    TTFT p99:  {p99:.3f}s")
        print(f"    TTFT mean: {statistics.mean(ttft_times):.3f}s")
    return ttft_times

print("=== Baseline Benchmark (Random Routing, No Prefix Cache) ===")
baseline_ttft = run_benchmark("Baseline", GATEWAY_IP, questions, SYSTEM_PROMPT)

## Configuration 2: Prefix-Cache-Aware Routing

The first optimization is **prefix-cache-aware routing**. When vLLM finishes
a request, it keeps the KV-cache in GPU memory (if space allows). The EPP
tracks which prefixes are cached on which replicas. When a new request arrives,
the EPP routes it to the replica most likely to have a matching prefix.

This avoids redundant prefill computation. If 100 chatbot requests share the
same system prompt, only the first request on each replica does a full prefill.
The rest skip directly to processing the user message.

**Expected improvement: 30-60% TTFT reduction** for workloads with shared prefixes.

In [ ]:
# Enable prefix-cache-aware routing
prefix_cache_config = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-config
  namespace: llm-d-kvcache
data:
  config.yaml: |
    scoring:
      - name: prefix-cache-scorer
        weight: 0.7
      - name: load-aware-scorer
        weight: 0.3
    prefixCache:
      enabled: true
      indexer: hash
"""

with open("/tmp/prefix-cache-config.yaml", "w") as f:
    f.write(prefix_cache_config)

!kubectl apply -f /tmp/prefix-cache-config.yaml
!kubectl rollout restart deployment/epp -n $NAMESPACE
!kubectl rollout status deployment/epp -n $NAMESPACE --timeout=120s

# Wait for the EPP to start tracking caches
print("\nPrefix-cache-aware routing enabled.")
print("Warming up caches with initial requests...")

# Send a warm-up request so the system prompt gets cached
!curl -s http://$GATEWAY_IP:8080/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{"model": "Qwen/Qwen3-32B", "messages": [{"role": "system", "content": "You are a helpful AI assistant specializing in cloud-native infrastructure. You provide clear, concise answers about Kubernetes, containers, and distributed systems. Always include practical examples when possible."}, {"role": "user", "content": "Hello"}], "max_tokens": 10}' \
    > /dev/null 2>&1

time.sleep(2)
print("Cache warmed. Running benchmark...")
print()

print("=== Prefix-Cache-Aware Routing Benchmark ===")
prefix_cache_ttft = run_benchmark("Prefix-Cache Routing", GATEWAY_IP, questions, SYSTEM_PROMPT)

## Configuration 3: Precise Prefix Cache Indexer

The default hash-based prefix cache indexer uses a hash of the prompt tokens
to determine cache affinity. This works well for exact prefix matches but
misses partial overlaps.

The **precise prefix cache indexer** tracks cache state at the block level
(typically 16-token blocks in vLLM). It knows exactly how many blocks of a
given prefix are cached on each replica. This enables:

- **Partial prefix matching** - if 80% of the prefix is cached, the EPP knows
  and can still route there instead of choosing a cold replica
- **Better multi-turn routing** - as conversations grow, the indexer tracks
  the progressively longer prefix on each replica

**Expected improvement: 10-20% additional TTFT reduction** on top of basic
prefix-cache routing, especially for multi-turn conversations.

In [ ]:
# Enable the precise prefix cache indexer
precise_config = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-config
  namespace: llm-d-kvcache
data:
  config.yaml: |
    scoring:
      - name: prefix-cache-scorer
        weight: 0.7
      - name: load-aware-scorer
        weight: 0.3
    prefixCache:
      enabled: true
      indexer: precise
      blockSize: 16
      reportInterval: 1s
"""

with open("/tmp/precise-cache-config.yaml", "w") as f:
    f.write(precise_config)

!kubectl apply -f /tmp/precise-cache-config.yaml
!kubectl rollout restart deployment/epp -n $NAMESPACE
!kubectl rollout status deployment/epp -n $NAMESPACE --timeout=120s

# Warm up
!curl -s http://$GATEWAY_IP:8080/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{"model": "Qwen/Qwen3-32B", "messages": [{"role": "system", "content": "You are a helpful AI assistant specializing in cloud-native infrastructure. You provide clear, concise answers about Kubernetes, containers, and distributed systems. Always include practical examples when possible."}, {"role": "user", "content": "Hello"}], "max_tokens": 10}' \
    > /dev/null 2>&1

time.sleep(2)
print("Precise indexer enabled. Running benchmark...")
print()

print("=== Precise Prefix Cache Indexer Benchmark ===")
precise_ttft = run_benchmark("Precise Indexer", GATEWAY_IP, questions, SYSTEM_PROMPT)

## Configuration 4: CPU Offloading Tier

GPU memory is expensive and limited. When the KV-cache fills up, vLLM must
either evict cached prefixes or reject new requests. **CPU offloading** adds
a second tier: cold KV-cache entries are moved from GPU memory to CPU (host)
memory.

This is useful because:

- CPU memory is 10-20x larger than GPU memory (e.g., 512 GB vs 80 GB)
- Moving data from CPU to GPU via PCIe takes ~1-5 ms per block, much faster
  than recomputing the prefill (which takes 50-200 ms for long prompts)
- You effectively get a much larger prefix cache without adding GPUs

**Expected improvement:** 2-4x more concurrent users per GPU with only a
small latency increase for cache-miss requests.

In [ ]:
# Enable CPU offloading tier
# This requires configuring vLLM with the cpu-offload flag

cpu_offload_patch = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm
  namespace: llm-d-kvcache
spec:
  template:
    spec:
      containers:
        - name: vllm
          args:
            - "--model"
            - "Qwen/Qwen3-32B"
            - "--enable-prefix-caching"
            - "--cpu-offload-gb"
            - "64"
          resources:
            requests:
              memory: "128Gi"
            limits:
              memory: "128Gi"
"""

with open("/tmp/cpu-offload-patch.yaml", "w") as f:
    f.write(cpu_offload_patch)

print("Enabling CPU offloading tier (64 GB of host memory for KV-cache)...")
!kubectl apply -f /tmp/cpu-offload-patch.yaml
!kubectl rollout status deployment/vllm -n $NAMESPACE --timeout=600s

print("\nCPU offloading enabled.")
print("KV-cache tiers:")
print("  Tier 1: GPU memory (80 GB) - hot cache, zero-copy access")
print("  Tier 2: CPU memory (64 GB) - warm cache, ~1-5 ms to load")
print()

# Measure with CPU offloading
print("=== CPU Offloading Benchmark ===")
print("(Sending more requests to fill GPU cache and trigger offloading)")
print()

# Send extra requests to fill the GPU cache, forcing some entries to CPU
extended_questions = questions * 3  # 60 requests total
cpu_offload_ttft = run_benchmark("CPU Offload", GATEWAY_IP, extended_questions[:20], SYSTEM_PROMPT)

## Configuration 5: Disk Offloading Tier

For even more capacity, llm-d supports **disk offloading**. Cold KV-cache
entries that have been evicted from both GPU and CPU memory are written to
local NVMe storage.

The tiered hierarchy becomes:

```
Tier 1: GPU Memory   (80 GB)    ~0 ms access
Tier 2: CPU Memory   (64 GB)    ~1-5 ms access
Tier 3: NVMe Disk    (1+ TB)    ~5-20 ms access
```

Even the slowest tier (NVMe) is faster than recomputing a long prefill.
For a 4K-token system prompt, recomputing the prefill takes ~100 ms,
while loading from NVMe takes ~10 ms.

**Expected improvement:** 10-50x prefix cache capacity with graceful
latency degradation.

In [ ]:
# Enable disk offloading tier
# This requires a PersistentVolumeClaim for local NVMe storage

disk_offload_config = """apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: kv-cache-disk
  namespace: llm-d-kvcache
spec:
  accessModes: ["ReadWriteOnce"]
  storageClassName: local-nvme
  resources:
    requests:
      storage: 500Gi
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm
  namespace: llm-d-kvcache
spec:
  template:
    spec:
      containers:
        - name: vllm
          args:
            - "--model"
            - "Qwen/Qwen3-32B"
            - "--enable-prefix-caching"
            - "--cpu-offload-gb"
            - "64"
            - "--disk-offload-path"
            - "/cache/kv-disk"
            - "--disk-offload-gb"
            - "500"
          volumeMounts:
            - name: kv-disk
              mountPath: /cache/kv-disk
      volumes:
        - name: kv-disk
          persistentVolumeClaim:
            claimName: kv-cache-disk
"""

with open("/tmp/disk-offload-config.yaml", "w") as f:
    f.write(disk_offload_config)

print("Enabling disk offloading tier (500 GB NVMe)...")
!kubectl apply -f /tmp/disk-offload-config.yaml
!kubectl rollout status deployment/vllm -n $NAMESPACE --timeout=600s

print("\nDisk offloading enabled.")
print("KV-cache tiers:")
print("  Tier 1: GPU memory (80 GB)   - ~0 ms access")
print("  Tier 2: CPU memory (64 GB)   - ~1-5 ms access")
print("  Tier 3: NVMe disk  (500 GB)  - ~5-20 ms access")
print(f"  Total cache capacity: ~644 GB")
print()

print("=== Disk Offloading Benchmark ===")
disk_offload_ttft = run_benchmark("Disk Offload", GATEWAY_IP, questions, SYSTEM_PROMPT)

## Side-by-Side Comparison

Now let's compare all configurations we tested.

In [ ]:
# Compare all configurations
import statistics

configs = [
    ("1. Baseline (random)", baseline_ttft),
    ("2. Prefix-cache routing", prefix_cache_ttft),
    ("3. Precise indexer", precise_ttft),
    ("4. + CPU offloading", cpu_offload_ttft),
    ("5. + Disk offloading", disk_offload_ttft),
]

print("=" * 75)
print(f"{'Configuration':<30} {'TTFT p50':>10} {'TTFT p99':>10} {'Improvement':>12}")
print("=" * 75)

baseline_p50 = statistics.median(baseline_ttft) if baseline_ttft else 1.0

for name, ttft_data in configs:
    if ttft_data:
        p50 = statistics.median(ttft_data)
        p99 = sorted(ttft_data)[int(len(ttft_data) * 0.99)]
        improvement = ((baseline_p50 - p50) / baseline_p50) * 100
        imp_str = f"{improvement:+.0f}%" if name != "1. Baseline (random)" else "---"
        print(f"{name:<30} {p50:>9.3f}s {p99:>9.3f}s {imp_str:>12}")
    else:
        print(f"{name:<30} {'N/A':>10} {'N/A':>10} {'N/A':>12}")

print("=" * 75)
print()
print("Notes:")
print("- TTFT improvement is relative to the baseline.")
print("- CPU and disk offloading primarily improve capacity (concurrent users)")
print("  rather than latency. Their TTFT may be similar to prefix-cache routing")
print("  under low load but they shine under high concurrency.")

## Recommendations by Workload Type

Not every workload needs every optimization. Use this guide to pick the
right configuration for your use case.

### Chatbot / Conversational AI

- **Must have:** Prefix-cache-aware routing + precise indexer
- **Why:** All conversations share the same system prompt. Prefix caching
  avoids recomputing it on every request. The precise indexer also helps
  with multi-turn conversations where the context grows incrementally.
- **Optional:** CPU offloading if you serve many unique conversations
  and want to keep more of them warm.

### RAG (Retrieval-Augmented Generation)

- **Must have:** Prefix-cache-aware routing
- **Why:** Many RAG queries share the same instructions prefix. The
  retrieved context differs per query, so full prefix reuse is rare,
  but partial prefix matching with the precise indexer still helps.
- **Recommended:** CPU offloading. RAG prompts tend to be long (4K-16K
  tokens) and consume lots of KV-cache. Offloading expands capacity.

### Batch Processing / Document Analysis

- **Must have:** CPU + disk offloading
- **Why:** Batch jobs process many documents sequentially. Each document
  has a unique prompt, so prefix cache hit rates are low. But the sheer
  volume of concurrent requests demands maximum KV-cache capacity.
- **Optional:** Prefix-cache routing helps if documents share a common
  instruction preamble.

### Code Completion / IDE Integration

- **Must have:** Prefix-cache-aware routing + precise indexer
- **Why:** IDE requests have very high prefix overlap (same file contents
  as context, with only the cursor position changing). The precise indexer
  excels here because it can match partial overlaps.
- **Recommended:** CPU offloading for large codebases where each file
  creates a unique cache entry.

### Summary Table

| Workload | Prefix Routing | Precise Indexer | CPU Offload | Disk Offload |
|----------|:-:|:-:|:-:|:-:|
| Chatbot | Required | Recommended | Optional | Rarely needed |
| RAG | Required | Recommended | Recommended | Optional |
| Batch | Optional | Optional | Required | Recommended |
| Code Completion | Required | Required | Recommended | Optional |